In [ ]:
import os
os.environ["HF_TOKEN"] = "token"  # paste your actual token here
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

In [ ]:
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

from datasets import load_dataset
dataset = load_dataset("PranavaKailash/CyNER2.0_augmented_dataset")

df_train = dataset["train"].to_pandas()
print(df_train.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/241k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/236k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7751 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1661 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1662 [00:00<?, ? examples/s]

(7751, 2)


In [ ]:
df_train.head()

,tokens,ner_tags
0,"[We, believe, the, TelePort, Crew, Threat, Act...","[16, 16, 16, 6, 14, 14, 14, 16, 16, 16, 16, 2,..."
1,"[The, group, behind, the, OilRig, campaign, co...","[16, 6, 16, 6, 14, 14, 16, 16, 16, 1, 9, 16, 3..."
2,"[Its, major, functionality, is, also, implemen...","[16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 1..."
3,"[A, backdoor, also, known, as:, Trojan.WebToos...","[16, 3, 16, 16, 16, 1, 1, 1, 1, 1, 1, 1, 1, 1,..."
4,"[A, short, ,, constant, string, of, characters...","[16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 1..."


In [ ]:
CYNER_TAG_MAP = {
    0:  "O",
    1:  "B-Malware",
    2:  "I-Malware",
    3:  "B-Indicator",
    4:  "I-Indicator",
    5:  "B-System",
    6:  "I-System",
    7:  "B-Organization",
    8:  "I-Organization",
    9:  "B-Vulnerability",
    10: "I-Vulnerability",
    16: "O"
}

CYNER_ENTITIES = ["Malware", "Indicator", "System", "Organization", "Vulnerability"]

df_train["bio_labels"] = df_train["ner_tags"].apply(
    lambda tags: [CYNER_TAG_MAP.get(t, "O") for t in tags]
)

print(f"Dataset loaded: {df_train.shape[0]} rows")

Dataset loaded: 7751 rows


In [ ]:
# CELL 2: Load bert-base-NER pipeline

from transformers import pipeline

ner_pipeline = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=0  # change to -1 if no GPU
)

BERT_TO_CYNER = {
    "ORG":  "Organization",
    "PER":  "Organization",
    "LOC":  "System",
    "MISC": "Malware",
}

print("Model loaded.")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded.


In [ ]:
# CELL 3: Run bert-base-NER on full dataset

def hf_predict_row(row):
    tokens_truncated = row["tokens"][:400]
    text = " ".join(tokens_truncated)
    preds = ner_pipeline(text)

    pred_labels = ["O"] * len(tokens_truncated)
    pred_labels += ["O"] * (len(row["tokens"]) - len(tokens_truncated))

    char = 0
    token_char_map = []
    for tok in tokens_truncated:
        token_char_map.append((char, char + len(tok)))
        char += len(tok) + 1

    for ent in preds:
        cyner_type = BERT_TO_CYNER.get(ent["entity_group"], None)
        if cyner_type is None:
            continue
        first = True
        for idx, (ts, te) in enumerate(token_char_map):
            if ts >= ent["start"] and te <= ent["end"]:
                pred_labels[idx] = f"{'B' if first else 'I'}-{cyner_type}"
                first = False

    return pred_labels

print("Running on full dataset — takes ~5-10 min on GPU...")
df_train["hf_pred"] = df_train.apply(hf_predict_row, axis=1)
print("Done.")

Running on full dataset — takes ~5-10 min on GPU...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Done.


In [ ]:
# CELL 4: Error analysis + print all results

from collections import defaultdict
from sklearn.metrics import classification_report

def flatten(label_lists):
    return [label for row in label_lists for label in row]

gold_flat = flatten(df_train["bio_labels"].tolist())
hf_flat   = flatten(df_train["hf_pred"].tolist())

tp = defaultdict(int)
fp = defaultdict(int)
fn = defaultdict(int)
wrong_label = defaultdict(int)

for g, p in zip(gold_flat, hf_flat):
    g_type = g.split("-")[1] if g != "O" else "O"
    p_type = p.split("-")[1] if p != "O" else "O"

    if g_type == "O" and p_type == "O":
        continue
    elif g_type != "O" and p_type == g_type:
        tp[g_type] += 1
    elif g_type != "O" and p_type == "O":
        fn[g_type] += 1
    elif g_type == "O" and p_type != "O":
        fp[p_type] += 1
    elif g_type != "O" and p_type != "O" and g_type != p_type:
        wrong_label[f"{g_type}→{p_type}"] += 1
        fn[g_type] += 1
        fp[p_type] += 1

print("=" * 55)
print(" bert-base-NER — Per-Entity Results")
print("=" * 55)
print(f"  {'Entity':<18} {'TP':>6} {'FP':>6} {'FN':>6}  {'Prec':>6} {'Rec':>6} {'F1':>6}")
print("  " + "─" * 52)

for etype in CYNER_ENTITIES:
    t, f_p, f_n = tp[etype], fp[etype], fn[etype]
    prec = t / (t + f_p) if (t + f_p) > 0 else 0.0
    rec  = t / (t + f_n) if (t + f_n) > 0 else 0.0
    f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
    print(f"  {etype:<18} {t:>6} {f_p:>6} {f_n:>6}  {prec:>6.3f} {rec:>6.3f} {f1:>6.3f}")

print("\nWrong Label Confusions:")
if wrong_label:
    for k, v in sorted(wrong_label.items(), key=lambda x: -x[1]):
        print(f"  {k}: {v}")
else:
    print("  None")

print("\nFull Classification Report:")
print(classification_report(
    gold_flat, hf_flat,
    labels=[f"B-{e}" for e in CYNER_ENTITIES] + [f"I-{e}" for e in CYNER_ENTITIES],
    zero_division=0
))

 bert-base-NER — Per-Entity Results
  Entity                 TP     FP     FN    Prec    Rec     F1
  ────────────────────────────────────────────────────
  Malware               159   3096  41309   0.049  0.004  0.007
  Indicator               0      0   7312   0.000  0.000  0.000
  System                  2    547   2924   0.004  0.001  0.001
  Organization            6   3549    915   0.002  0.007  0.003
  Vulnerability           0      0   4576   0.000  0.000  0.000

Wrong Label Confusions:
  Indicator→Organization: 1156
  System→Malware: 793
  Indicator→Malware: 525
  System→Organization: 345
  Vulnerability→Malware: 191
  Malware→System: 157
  Vulnerability→System: 123
  Vulnerability→Organization: 52
  Malware→Organization: 32
  Organization→Malware: 24
  Indicator→System: 18

Full Classification Report:
                 precision    recall  f1-score   support

      B-Malware       0.05      0.00      0.01     41046
    B-Indicator       0.00      0.00      0.00      5471
     

In [ ]:
# CELL 5: Sample FP and FN examples

print("=== Sample FALSE POSITIVES (predicted entity, gold=O) ===\n")
count = 0
for _, row in df_train.iterrows():
    for i, (g, p) in enumerate(zip(row["bio_labels"], row["hf_pred"])):
        if g == "O" and p != "O":
            context = " ".join(row["tokens"][max(0,i-3):i+4])
            print(f"  Token: '{row['tokens'][i]}'  predicted: {p}")
            print(f"  Context: ...{context}...")
            print()
            count += 1
            if count >= 5:
                break
    if count >= 5:
        break

print("=== Sample FALSE NEGATIVES (missed entity) ===\n")
count = 0
for _, row in df_train.iterrows():
    for i, (g, p) in enumerate(zip(row["bio_labels"], row["hf_pred"])):
        if g != "O" and p == "O":
            context = " ".join(row["tokens"][max(0,i-3):i+4])
            print(f"  Token: '{row['tokens'][i]}'  gold: {g}")
            print(f"  Context: ...{context}...")
            print()
            count += 1
            if count >= 5:
                break
    if count >= 5:
        break

=== Sample FALSE POSITIVES (predicted entity, gold=O) ===

  Token: 'Crew'  predicted: I-Organization
  Context: ...believe the TelePort Crew Threat Actor is...

  Token: 'Threat'  predicted: I-Organization
  Context: ...the TelePort Crew Threat Actor is operating...

  Token: 'Actor'  predicted: I-Organization
  Context: ...TelePort Crew Threat Actor is operating out...

  Token: 'OilRig'  predicted: B-Organization
  Context: ...group behind the OilRig campaign continues to...

  Token: 'Microsoft'  predicted: B-Malware
  Context: ...emails with malicious Microsoft Excel documents to...

=== Sample FALSE NEGATIVES (missed entity) ===

  Token: 'groups'  gold: I-System
  Context: ...Europe with the groups major motivations appearing...

  Token: 'financial'  gold: I-Indicator
  Context: ...appearing to be financial in nature through...

  Token: 'cybercrime'  gold: I-System
  Context: ...in nature through cybercrime and/or corporate espionage....

  Token: 'corporate'  gold: I-System
 

In [ ]:
# Export 10 random rows as CSV for Batch 1 annotation

batch1 = df_train.sample(n=10, random_state=42).reset_index(drop=False)
batch1.to_csv("batch1_10docs.csv", index=False)
print("Saved: batch1_10docs.csv")
batch1[["index", "tokens", "bio_labels"]].head(10)

Saved: batch1_10docs.csv


,index,tokens,bio_labels
0,5081,"[So, the, system, doesn, ’, t, see, any, stran...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
1,5103,"[A, backdoor, also, known, as:, Trojanpws.Win6...","[O, B-Indicator, O, O, O, B-Malware, B-Malware..."
2,4381,"[ALLMSG, –, send, C, &, C, all, SMSs, received...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3,1559,"[The, actor, also, built, solid, backend, infr...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]"
4,3891,"[While, we, do, not, have, complete, targeting...","[O, O, O, O, O, O, O, O, O, O, O, O, B-Indicat..."
5,1322,"[It, was, hosting, an, Adobe, Flash, exploit, ...","[O, O, O, O, B-System, O, B-Indicator, O, O, O..."
6,4639,"[A, backdoor, also, known, as:, Exploit.Win64,...","[O, B-Indicator, O, O, O, B-Malware, B-Malware..."
7,7662,"[Brain, Test, has, been, removed, from, Google...","[B-Indicator, O, O, O, O, O, B-System, O, O, O..."
8,4129,"[Figure, 5, .]","[O, O, O]"
9,408,"[Hashes, of, samples, Type, Package, name, SHA...","[O, O, O, O, O, O, O, O, O, O, B-Malware, B-Ma..."


In [ ]:
import pandas as pd
import ast
import json
import re
from sklearn.metrics import cohen_kappa_score, f1_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv("batch 1.csv")
print(f"Rows loaded: {len(df)}")

Rows loaded: 20


In [ ]:
def parse_tokens(tok_str):
    tok_str = str(tok_str).strip()
    tokens = re.findall(r"'((?:[^'\\]|\\.)*)'", tok_str)
    if tokens:
        return tokens
    return tok_str.strip("[]").split()

df["tokens_parsed"] = df["tokens"].apply(parse_tokens)

print("tokens_parsed check:")
for i in range(3):
    parsed = df["tokens_parsed"].iloc[i]
    print(f"  Row {i}: {parsed[:6]}  (length={len(parsed)})")

tokens_parsed check:
  Row 0: ['So', 'the', 'system', 'doesn', '’', 't']  (length=19)
  Row 1: ['So', 'the', 'system', 'doesn', '’', 't']  (length=19)
  Row 2: ['A', 'backdoor', 'also', 'known', 'as:', 'Trojanpws.Win64']  (length=16)


In [ ]:
def entities_to_bio(tokens, entities_str):
    try:
        entities = json.loads(entities_str)
    except:
        return ["O"] * len(tokens)

    char = 0
    token_spans = []
    for tok in tokens:
        token_spans.append((char, char + len(tok)))
        char += len(tok) + 1

    labels = ["O"] * len(tokens)

    for ent in entities:
        start = ent["start"]
        end   = ent["end"]
        tag   = ent["labels"][0]
        first = True
        for idx, (ts, te) in enumerate(token_spans):
            if ts >= start and te <= end:
                labels[idx] = f"{'B' if first else 'I'}-{tag}"
                first = False

    return labels

df["anno_labels"] = df.apply(
    lambda row: entities_to_bio(row["tokens_parsed"], row["entities"]), axis=1
)

print("anno_labels check:")
for i in range(4):
    print(f"  Row {i}: {df['anno_labels'].iloc[i]}")

anno_labels check:
  Row 0: ['O', 'O', 'O', 'B-System', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Vulnerability', 'I-Vulnerability', 'I-Vulnerability', 'O', 'O', 'O', 'O', 'O', 'O']
  Row 1: ['O', 'O', 'O', 'B-System', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Vulnerability', 'O', 'O', 'O', 'O', 'O', 'O']
  Row 2: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Malware', 'I-Malware', 'O', 'O', 'O', 'B-Malware', 'O', 'O']
  Row 3: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Malware', 'O', 'O', 'O', 'B-Malware', 'O', 'O']


In [ ]:
ANNOTATOR_A = 152634
ANNOTATOR_B = 152635

df_a = df[df["annotator"] == ANNOTATOR_A].sort_values("id").reset_index(drop=True)
df_b = df[df["annotator"] == ANNOTATOR_B].sort_values("id").reset_index(drop=True)

shared_ids = sorted(set(df_a["id"]) & set(df_b["id"]))
df_a = df_a[df_a["id"].isin(shared_ids)].sort_values("id").reset_index(drop=True)
df_b = df_b[df_b["id"].isin(shared_ids)].sort_values("id").reset_index(drop=True)

print(f"Annotator A: {len(df_a)} docs")
print(f"Annotator B: {len(df_b)} docs")
print(f"Shared docs: {len(shared_ids)}")

Annotator A: 10 docs
Annotator B: 10 docs
Shared docs: 10


In [ ]:
flat_a = [l for row in df_a["anno_labels"] for l in row]
flat_b = [l for row in df_b["anno_labels"] for l in row]

print(f"Annotator A tokens: {len(flat_a)}")
print(f"Annotator B tokens: {len(flat_b)}")

assert len(flat_a) == len(flat_b), f"Mismatch: A={len(flat_a)}, B={len(flat_b)}"

Annotator A tokens: 200
Annotator B tokens: 200


In [ ]:
binary_a = [0 if l == "O" else 1 for l in flat_a]
binary_b = [0 if l == "O" else 1 for l in flat_b]

kappa = cohen_kappa_score(binary_a, binary_b)
f1    = f1_score(binary_a, binary_b, zero_division=0)

print("=== Overall IAA — Batch 1 ===")
print(f"  Cohen's Kappa : {kappa:.3f}")
print(f"  Entity F1     : {f1:.3f}")

if   kappa < 0.40: print("Poor — major guideline revision needed")
elif kappa < 0.60: print("Moderate — significant revision needed")
elif kappa < 0.80: print("Substantial — acceptable (target ≥ 0.70)")
else:              print("Excellent — proceed with training!")

=== Overall IAA — Batch 1 ===
  Cohen's Kappa : 0.748
  Entity F1     : 0.795
Substantial — acceptable (target ≥ 0.70)


In [ ]:
CYNER_ENTITIES = ["Malware", "Indicator", "System", "Organization", "Vulnerability"]

print("=== Per-Entity IAA ===")
print(f"  {'Entity':<18} {'Kappa':>6}  {'F1':>6}  Status")
print("  " + "─" * 50)

for etype in CYNER_ENTITIES:
    a_bin = [1 if etype in l else 0 for l in flat_a]
    b_bin = [1 if etype in l else 0 for l in flat_b]

    if sum(a_bin) + sum(b_bin) == 0:
        print(f"  {etype:<18} {'—':>6}  {'—':>6}  (no instances)")
        continue

    k = cohen_kappa_score(a_bin, b_bin)
    f = f1_score(a_bin, b_bin, zero_division=0)

    if   k < 0.40: status = "Poor"
    elif k < 0.70: status = "Below threshold"
    elif k < 0.80: status = "Acceptable"
    else:          status = "Excellent"

    print(f"  {etype:<18} {k:>6.3f}  {f:>6.3f}  {status}")

=== Per-Entity IAA ===
  Entity              Kappa      F1  Status
  ──────────────────────────────────────────────────
  Malware             0.837   0.857  Excellent
  Indicator           0.000   0.000  Poor
  System              0.174   0.182  Poor
  Organization        0.000   0.000  Poor
  Vulnerability       0.444   0.462  Below threshold


In [ ]:
doc_agreement = df.groupby("id")["agreement"].first().reset_index()
doc_agreement.columns = ["doc_id", "agreement_pct"]
doc_agreement = doc_agreement.sort_values("agreement_pct")

print("=== Per-Document Agreement ===\n")
print(doc_agreement.to_string(index=False))
print(f"\nMean : {doc_agreement['agreement_pct'].mean():.1f}%")
print(f"Min  : {doc_agreement['agreement_pct'].min():.1f}%")
print(f"Max  : {doc_agreement['agreement_pct'].max():.1f}%")

=== Per-Document Agreement ===

   doc_id  agreement_pct
261673501           0.00
261673500           0.00
261673502          20.00
261673503          36.36
261673507          61.90
261673505          66.67
261673499          77.77
261673498          80.95
261673504          98.60
261673506         100.00

Mean : 54.2%
Min  : 0.0%
Max  : 100.0%


##Refinement Plan

- Clearly define and disucss indicators, systems, and organizations when before looking at batch 2
- Use examples and definitions in the batch 2 labeling.
- Define context-dependency rules for Indicator
- Standardize multi-token tagging
Narrow the Organization defintion

In [ ]:
df.head()

,agreement,annotation_id,annotator,bio_labels,created_at,entities,hf_pred,id,index,lead_time,ner_tags,spacy_pred,tokens,updated_at,tokens_parsed,anno_labels
0,80.95,93305871,152634,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...",2026-04-13T03:48:04.179106Z,"[{""end"":19,""text"":""system"",""start"":13,""labels""...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...",261673498,5081,44.813,[16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 ...,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...",['So' 'the' 'system' 'doesn' '’' 't' 'see' 'an...,2026-04-13T03:48:04.179117Z,"[So, the, system, doesn, ’, t, see, any, stran...","[O, O, O, B-System, O, O, O, O, O, O, B-Vulner..."
1,80.95,93305715,152635,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...",2026-04-13T03:43:25.314016Z,"[{""end"":19,""text"":""system"",""start"":13,""labels""...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...",261673498,5081,36.605,[16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 ...,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...",['So' 'the' 'system' 'doesn' '’' 't' 'see' 'an...,2026-04-13T03:44:03.485808Z,"[So, the, system, doesn, ’, t, see, any, stran...","[O, O, O, B-System, O, O, O, O, O, O, O, O, B-..."
2,77.77,93305874,152634,"['O', 'B-Indicator', 'O', 'O', 'O', 'B-Malware...",2026-04-13T03:48:07.837003Z,"[{""end"":53,""text"":""Trojanpws.Win64"",""start"":38...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...",261673499,5103,42.492,[16 3 16 16 16 1 1 1 1 1 1 1 1 1 1 1],"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...",['A' 'backdoor' 'also' 'known' 'as:' 'Trojanpw...,2026-04-13T03:48:07.837015Z,"[A, backdoor, also, known, as:, Trojanpws.Win6...","[O, O, O, O, O, O, O, O, B-Malware, I-Malware,..."
3,77.77,93305798,152635,"['O', 'B-Indicator', 'O', 'O', 'O', 'B-Malware...",2026-04-13T03:46:10.584202Z,"[{""end"":53,""text"":""Trojanpws.Win64"",""start"":38...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...",261673499,5103,84.537,[16 3 16 16 16 1 1 1 1 1 1 1 1 1 1 1],"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...",['A' 'backdoor' 'also' 'known' 'as:' 'Trojanpw...,2026-04-13T03:46:10.584215Z,"[A, backdoor, also, known, as:, Trojanpws.Win6...","[O, O, O, O, O, O, O, O, O, B-Malware, O, O, O..."
4,0.00,93305888,152634,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...",2026-04-13T03:48:22.269675Z,"[{""end"":44,""text"":""SMSs"",""start"":40,""labels"":[...","['B-Organization', 'O', 'O', 'B-Organization',...",261673500,4381,38.902,[16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 ...,"['O', 'O', 'O', 'B-Organization', 'I-Organizat...",['ALLMSG' '–' 'send' 'C' '&' 'C' 'all' 'SMSs' ...,2026-04-13T03:48:22.269686Z,"[ALLMSG, –, send, C, &, C, all, SMSs, received...","[O, B-Malware, O, O, O, O, O, O, O, O, O, O, O..."


In [ ]:
# df_train — readable sentences
df_train["review_text"] = df_train["tokens"].apply(lambda tokens: " ".join(tokens))

print("df_train review_text check:")
for i in range(3):
    print(f"  Row {i}: {df_train['review_text'].iloc[i]}")
    print()

df_train review_text check:
  Row 0: We believe the TelePort Crew Threat Actor is operating out of Russia or Eastern Europe with the groups major motivations appearing to be financial in nature through cybercrime and/or corporate espionage.

  Row 1: The group behind the OilRig campaign continues to leverage spear-phishing emails with malicious Microsoft Excel documents to compromise victims.

  Row 2: Its major functionality is also implemented through the call of the asynchronous task ( “ org.starsizew.i ” ) , including uploading the incoming SMS messages to the remote C2 server and executing any commands as instructed by the remote attacker .



In [ ]:
df_train.head()

,tokens,ner_tags,bio_labels,hf_pred,review_text
0,"[We, believe, the, TelePort, Crew, Threat, Act...","[16, 16, 16, 6, 14, 14, 14, 16, 16, 16, 16, 2,...","[O, O, O, I-System, O, O, O, O, O, O, O, I-Mal...","[O, O, O, B-Organization, I-Organization, I-Or...",We believe the TelePort Crew Threat Actor is o...
1,"[The, group, behind, the, OilRig, campaign, co...","[16, 6, 16, 6, 14, 14, 16, 16, 16, 1, 9, 16, 3...","[O, I-System, O, I-System, O, O, O, O, O, B-Ma...","[O, O, O, O, B-Organization, O, O, O, O, O, O,...",The group behind the OilRig campaign continues...
2,"[Its, major, functionality, is, also, implemen...","[16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 1...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Its major functionality is also implemented th...
3,"[A, backdoor, also, known, as:, Trojan.WebToos...","[16, 3, 16, 16, 16, 1, 1, 1, 1, 1, 1, 1, 1, 1,...","[O, B-Indicator, O, O, O, B-Malware, B-Malware...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",A backdoor also known as: Trojan.WebToos.A6 Tr...
4,"[A, short, ,, constant, string, of, characters...","[16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 1...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","A short , constant string of characters is ins..."


In [ ]:
# Export 200 random rows as CSV for Batch 2 annotation

batch1 = df_train.sample(n=210, random_state=42).reset_index(drop=False)
batch1.to_csv("batch2_210docs.csv", index=False)
print("Saved: batch2_210docs.csv")
batch1[["index", "review_text", "bio_labels"]].head(10)

Saved: batch2_210docs.csv


,index,review_text,bio_labels
0,5081,So the system doesn ’ t see any strange proces...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
1,5103,A backdoor also known as: Trojanpws.Win64 Win3...,"[O, B-Indicator, O, O, O, B-Malware, B-Malware..."
2,4381,ALLMSG – send C & C all SMSs received and sent...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3,1559,The actor also built solid backend infrastruct...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]"
4,3891,"While we do not have complete targeting , info...","[O, O, O, O, O, O, O, O, O, O, O, O, B-Indicat..."
5,1322,It was hosting an Adobe Flash exploit targetin...,"[O, O, O, O, B-System, O, B-Indicator, O, O, O..."
6,4639,A backdoor also known as: Exploit.Win64 Exploi...,"[O, B-Indicator, O, O, O, B-Malware, B-Malware..."
7,7662,Brain Test has been removed from Google Play s...,"[B-Indicator, O, O, O, O, O, B-System, O, O, O..."
8,4129,Figure 5 .,"[O, O, O]"
9,408,Hashes of samples Type Package name SHA256 dig...,"[O, O, O, O, O, O, O, O, O, O, B-Malware, B-Ma..."
